In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
df_main = pd.read_csv(r'FINAL_BINARY_DATASET.csv')
target_mapping = {'No_Disease':0,
                  'Disease':1}
df_main['Target'] = df_main['Target'].map(target_mapping)

In [ ]:
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


df_fe = df_main.copy()
df_fe["IL6_albumin_ratio"] = df_fe["Interleukin-6 (IL-6) level"] / df_fe["Serum albumin level"]
df_fe["protein_eGFR_ratio"] = df_fe["Urine protein-to-creatinine ratio"] / df_fe["Estimated Glomerular Filtration Rate (eGFR)"]
df_fe["CRP_WBC_ratio"] = df_fe["C-reactive protein (CRP) level"] / df_fe["White blood cell count (cells/cumm)"]
df_fe["log_Cystatin_C"] = np.log1p(df_fe["Cystatin C level"])
df_fe["sqrt_Cholesterol"] = np.sqrt(df_fe["Cholesterol level"])
df_fe["Age_group"] = pd.cut(df_fe["Age of the patient"], bins=[0, 30, 50, 70, 100],
                            labels=["<30", "30-50", "50-70", "70+"])
df_fe["BMI_category"] = pd.cut(df_fe["Body Mass Index (BMI)"], bins=[0, 18.5, 25, 30, np.inf],
                               labels=["Underweight", "Normal", "Overweight", "Obese"])
appetite_counts = df_fe["Appetite (good/poor)"].value_counts()
df_fe["Appetite_encoded"] = df_fe["Appetite (good/poor)"].map(appetite_counts)

CAT_boost_selected_features = [
    "Pus cells in urine", "Age of the patient", "Body Mass Index (BMI)", "Sodium level (mEq/L)",
    "Estimated Glomerular Filtration Rate (eGFR)", "Urine protein-to-creatinine ratio",
    "Duration of diabetes mellitus (years)", "C-reactive protein (CRP) level", "Sugar in urine",
    "Cystatin C level", "Interleukin-6 (IL-6) level", "White blood cell count (cells/cumm)",
    "Cholesterol level", "Serum phosphate level", "Serum albumin level",
    "Family history of chronic kidney disease", "Bacteria in urine", "Appetite (good/poor)"
]
for col in CAT_boost_selected_features:
    df_fe[col + "_missing_flag"] = df_fe[col].isnull().astype(int)

categorical_cols = [
    "Age_group", "BMI_category", "Appetite (good/poor)", "Family history of chronic kidney disease",
    "Pus cells in urine", "Sugar in urine", "Bacteria in urine"
]
for col in categorical_cols:
    le = LabelEncoder()
    df_fe[col] = le.fit_transform(df_fe[col].astype(str).fillna("missing"))


TARGET_COLUMN = "Target"
X = df_fe.drop(columns=[TARGET_COLUMN])
y = df_fe[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)


models = {
    'CatBoost': (
        CatBoostClassifier(verbose=False, random_state=42, early_stopping_rounds=100),
        {
            'depth': [4, 6],
            'learning_rate': [0.03, 0.1],
            'l2_leaf_reg': [3, 5],
            'iterations': [500, 1000]
        }
    ),
    'XGBoost': (
        XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        {
            'max_depth': [3, 6],
            'learning_rate': [0.03, 0.1],
            'n_estimators': [500, 1000],
            'reg_lambda': [1, 3]
        }
    ),
    'LogisticRegression': (
        Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(random_state=42, max_iter=500, class_weight='balanced'))
        ]),
        {
            'clf__C': [0.01, 0.1, 1, 10],
            'clf__penalty': ['l1', 'l2'],
            'clf__solver': ['liblinear']
        }
    ),
    'DecisionTree': (
        DecisionTreeClassifier(random_state=42, class_weight='balanced'),
        {
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5, 10]
        }
    ),
    'RandomForest': (
        RandomForestClassifier(random_state=42, class_weight='balanced'),
        {
            'n_estimators': [100, 200],
            'max_depth': [None, 10, 20]
        }
    ),
    'NaiveBayes': (
        GaussianNB(),
        {}  
    ),
    'AdaBoost': (
        AdaBoostClassifier(random_state=42),
        {
            'n_estimators': [50, 100],
            'learning_rate': [0.03, 0.1]
        }
    ),
    'GradientBoosting': (
        GradientBoostingClassifier(random_state=42),
        {
            'n_estimators': [100, 200],
            'learning_rate': [0.03, 0.1],
            'max_depth': [3, 6]
        }
    ),
    'KNN': (
        Pipeline([
            ('scaler', StandardScaler()),
            ('clf', KNeighborsClassifier())
        ]),
        {
            'clf__n_neighbors': [3, 5, 15],
            'clf__weights': ['uniform', 'distance'],
            'clf__metric': ['euclidean', 'manhattan']
        }
    )
}


for name, (model, param_grid) in models.items():
    print(f"\n Running GridSearchCV for {name}")
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=3,
        scoring='recall',
        n_jobs=-1
    )
    grid_search.fit(X_train_res, y_train_res)
    best_model = grid_search.best_estimator_
    print(f" Best parameters for {name}: {grid_search.best_params_}")

    probs = best_model.predict_proba(X_test)
    best_recall, best_threshold = 0, 0.5
    for t in np.linspace(0.2, 0.6, 100):
        preds = (probs[:, 1] >= t).astype(int)
        report = classification_report(y_test, preds, output_dict=True)
        recall_class_1 = report['1']['recall']
        accuracy = report['accuracy']
        if recall_class_1 > best_recall and accuracy >= 0.80:
            best_recall = recall_class_1
            best_threshold = t

    final_preds = (probs[:, 1] >= best_threshold).astype(int)
    print(f"\n {name} — Optimal Threshold: {best_threshold:.2f}")
    print(f" Recall (Class 1): {best_recall:.2f}")
    print(" Final Classification Report:")
    print(classification_report(y_test, final_preds))

    cm = confusion_matrix(y_test, final_preds)
    ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues')
    plt.title(f"Confusion Matrix - {name}")
    plt.tight_layout()
    plt.show()
